# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "d7e4b17dea0647d859bbeea5d7809099c5582f7ee2b3bde05aec385864aa862c",
    "gzip_base64": "H4sIAAAAAAAC/6U827LbRnLv5ysg7QOAc4YgD3VZm+dAVV5Fu3ElsVW2k0oVw0INgSGJFQjAuJyLFVXlH/KH+ZL0ZQYYgCDlLetBIgYzPd09fZ+GXr58+d37D7OfG1U626LNE5XM4iJv1FPjqKdSVelR5U3tyDxxjkrWbaUS4dTyWGZqJrN0n6vEqZu0iQ9pvg+urn5Sba1qpzkoJz6o+BO83sq6nidVe3Qei+qTqurA+ZjJfF5WxVY5idrJNmucpnDywknznapUHqvg6gelktrJ22P5DDvGKf2DKO7STAlntzuWai+cMmt5O/WUAiL53nn/7//0nVO1eQO4O7uicmYz9aTitgGgL1++vEqPZVE1jqz2paxqZZ4Psj5k6dY8/r0ucvP7KJvD1a4qjk4Jv2CSo198xBf6d31omzTrntot0Berur4yQ0SKI4Gmsptl6MHhesd7EK1BDbyVmdkoLqpKZbJBwtM8iUolP/WQVabipkrjaNdmWQSI7wkeMCNTcCTf/fLBCZ1Xy8VicfXTjz/+Ag+IuBdFuHUU+UGl6iJ7UJ4fAEvwwNe3m6urKzgbIEsu37z1kHB/deXAnyTdq7oJNb8CPcGnd49pc2DgtCAoSpV7brV1fcKoqZQ8MhT8g4ezzYr4E5y7kzaq8jJ53CZyxRMBLZl4t4vl62v8yxdb1/VXvH3Qlgmww6PlvHelmrbKNXrBQT3xL8CMKXmsYIsIz5WQEw8ya5WmCQfCHu+7RiFnZfUc4nOAZEV1u9ulT54bNMfS5S27aQEDR73xcIcggcOuPdpCwIEBT8OlkFlWPEa5zMO/yqxW/o37X/kJpEqVmYwVI6JRz+Rz0TZeUzQyE45W0PDbhQDRrPZpHr4FuZCJCoM338D7VtXhD0UOwqJyVe2f6UFTSijVoaeBCAYgcDVjku4cT+bPXgryUzcSdNF7ENuiyHwHjisvGmfwxkvzRuyyQjZ+NwH1JUhrkFRgivfg00E/4CHz7n4nAXrB7eI+1Ajdh28XCzO+uEe87hlHHORf97f4Wy+YLa/N6MLvZauSaa2c/8D9PlRVUXkuGhSzCHb8v//5X9ipNix8F97WzETE01toxvjI7krB20XtdgwacYHPBRjh9wRdo9rdh/SKaKKBS/j9XLRVrJxjWzcOGEaDoVMrQDoxu//J+R4o2CvgRZGozEnaSjZpkdeAdpy1gL4CW13KJEGD7PygHmCqeoqReDSTlfoVTqBBVsgyIJBH+ZQe22NIxwZHCchoNvmEtNaQJKzQXHnMGHrj3xF3Qg0BjgKmGSbhm/vbr9Nt6DFHA3g5mZIP4EYysJ+AtcyBG7VTMExHtklaaHbEgFIT3uJ+mtUaFzCKsA88eEtBhMUqzTw+KsbTnyM0X8ORpYzTJgXVWOPwDUxAoClKw3ohaJ/Z7YbBEhoo1PS6kvleeTTD3/TUI2Lh7cqGrHHjSeCRYGixuXPKIktjmmDe0G8CAz9B1BzUYL03jJhNRnranNXT5nfoadOrZa+Zi/vmnng2JzeCNDdIM2JBcPDHi7AGN6QSr1aNR28uKuJfUIzA1DnooetO4BkN4bR5CiKKzh5hUugBdKQg2Si/6N302Y8O4JYPyd5aHSUAzfeshjPk+BoOsZvQgCirJjTjNyzg3bLr/uzWKbyf1+3RG46tNn7PtqxA6fM6cFr4Be8+XjxYekjDY5r3S0c7axAsZLPUv9agewAgDVnx7pCuJgxf4ewgdku3EGdsDe8x1qvGFoE1cK7tbZYe08bYHYtdiOghFUhqVggeHHBBIDX2W1A4wlbopTx6sxzRUJRkycI1n0PDr4dCx4Teh4Mp9+Eh3dis0JBWtkSjwhHu+qX4pJ5DDjicZiW3tdfMDDFaJ59DF6K4B5C9JDKcczuYqI6jHUBYU4xL6jCDaNRjwSSG3NwKRHU+By9FirOeZuNmEuUe8BBrjx086C5SzQ9sJhb+TbBYLK8HdM2H/Ca7runEyITXR7mSVcQr3DFYdysztChJhMrb8wIFN5AlhHsJCrHfGzUzyhv5V+PZJNl6+NDmnzorSOotPCm2Ph6+gvhZVRjz/ZaWpCiCtIVUsD8GMHhVIwAwSeBCyBmaehI8dtNbNO09D4z/DIGVyYxWM5u6GZC+KJ0EsBMxS/wuaOnZQBQY0j67GPo9uatUuOgyIgLvrqR+hDnuaitgVtk25iUToMdoBvwlBmdm/XEpCogMSu5qAs0vg/j4s8vpW4S8dFdkoFxiTbSrJBhkd8WMcjVxEdiiyAQhKxM3Xg2xIIvRz9JRpYvxVD+KT8I1ihQZCXFX5hfsSQyEbejfLzr+xUQgIrc/SERkI0VdhfWOMwWK6mX2KJ/raJmEv1TgRZLmuVShS47v1bKP4OrqRUgeDYQM4WAaUypIe16ES+P78rJ3jjjHDyCA974eY/ISkERIZQvIuz7982+DmMUkKgBS04d+GhJkyBKiYudB/J/rXfoXIY4GoJKe2w+6oh8tIYkDhRWfv/in01xMwl2/I79/x5ECRLw8RbiYqcPUCT/yc1uW7JPpdCwg9YqyfOHoxTaZ/SyTh1EVIAL19vp3hl4btTBknFYakk5op+bxvv3J6MQYhyOqM1gxATN/+OYPEctkYS5dE1XW+endPLffL8D80PA5eJQP5jdl764/pK2Th45ItsOey9yISkDFgOzqAIifBj0Y65AFYcFM36sp5xAOZJdgbn5faklVA51fXgv7HLSUMel8zOH0adMMvTnvzdk3D1kFCUHDPMUapvU8Ghw/JSkwnQsXrPVUC4qKTzrP5tlVCuSAlcPCRairFno/mrBHP0cWMyLjGzL8uce22L3hyQHXAQJwmKrytEpx8SeIi/IZuWP4OgbZmx/e/uT9i3CA5iVT86Oe6NBKkFKMNRLn8YDlJD5frIZhhIcCZPIlyKxyQ5mrKUJJ0cnle5kXeRoDWLJYmHqgH0YIGBHPj+kTQi3y7Dlw/sboQ+iOdSvaCY1Q9aASDc5QA2dNOHFJDCGAn4oPulKIsRKCmoHfhHALS4ok3DMUboe0puZUtS+rBVWbe2uXy4Ag57O8qBvIefEnyT1yicbxr9QVIHvmuGFAVjzCYemkd4VJMaxd4vp4JeGfMj5G9e3bTPFa4qW/EVToJMFjHhPnQstn8cQ+tUPd6fJEo1H9WR9VI9E5hFRJAs+V1B6pQTfXJ4fH5SZ/kAOYtewCmN6Ipc0F6RqK/TB4vZCoGahOUsCR6dQxPtDhWUdM9QuUWytnIIKHWCXFY75VElILsd74NyOUwVxhdE7vhlBOkkyOApSJUD2KXyyqqDgY0hmsORqkGTMOxL/xVxgVZir3aIrPb2/M283YZ1AQ4UFUUP9a0b9HOFauQF5fL32ND6pKqAt2FuyJeptAKgTj3q81lc0axOooowdV1RieLkHkZNPWkJGgLqrEFZ2p4FMdmg6hz56Mta3op8I+kBIjIiyzgqMKbcWDjz99+PnDL8Iy+f3PU7iWdbORGRu9IAeui9PJk1SdbALCCPT9oK8OKoX3FBB4zTshC5z3VmFJF8lq7d1qXV7gCtSgcvY9hO4YxWOMxNHoHbnHNG/T5plvRNqaTOWvrcxwrM3htNJdqpJAa4BVczY2l86YHTY5djtOwAHtoU/44e2KDFJ2YUUXVLXW4/M+EpzmO2+3du0hyJxGPhUvfWLQsvASOG1PhgD14GZQJC25whTR5UX6oKKm0Pj2+1I2bF8zvAgNFpf83986tIzlMS4QwKkaDlt7F3AUw4gUNzGFdTCuEZJh5xWjy4Ae0zvSbcsqE3knppjR4ZsDDkvmE6y32aTVs7+GGbHMAmVzS5vxF+HkQVxiXu/mfy/vAFPeRecZkChNhnZdwOpra0H4kjsPOL8coz3IPS9i/ZG2JoCMtUlxnSTdgfbXHIhY4c6pFlmsnNC/0UttRLWwkHI/R2QMbG0UNNJF/Jg20BmDxVZH/hWFo/TAmmsLlFFmC8qEr0e4oZ7Z7XHXhQ282CSBGLpiUthd7tiv2adgZGDUHBwLMtiepE+8OzAjAfYtztCJjzyTWfoH7MnkNlwxAeB0AOtBHWUzLOKcrO2qnn3tBAB5A0hYfdnMpoD7X4GuadjLlMAGfz6dMirc9FSMXgzYPFIkOPsx9edxHsDRBoQBDCUC5Mkc28Vax3vKiylgz7EkOAehwAv4em5wn6PLdo5pTdHi0AjjPqa60+asVLVXFQVrXmTZY1vVtG217HY3+atZZ2+a7mwtjc5op97qsiO+sqoNuzhvsv4a3OiyG/QgAgwWXXMvLm/4XhwHrVtxBIP3cPEnD/8SPPCvP77/l+jDf/639fTDX3rlgHPYg17q7NxsbUY54DiZPPBlNoCBzSHbP3hL+XXt+VyL+PxlcKNv6iXaX+iK3maYbGAVm3IoI6hYJ92AhyXja/Df8eroM097SdNeblaLV8kXd1jGpnW6FtCl/31OZqVIsDVhpylanYSUsYTQO+kDKz1xDQs343hpBBudOS8/deHkvVeTzhgXkPPpnC8R7J7uMfBBvJPtg07mlxUEt55LbTmOiU41dLxKELusrQ/MqDsd3qoBlD85P6m4wFtkSZXNGnGVDTYsNI7cNfgipxC6qtoSC2aGYRz9BsMLjQJrzxhgdkmwLTsGbprr6z/Nt2CfFVtQGH4dXQ9MU5+mkTvVk2wfOnVU2uilLNIgyxjRW/qg39uKMPSZoe0zV5MlhMFpabzs47rr2aHf3m1hw09jke2mmavg0+20Z2DnUT43B0g5KIZFkwo6DOe4k2nmbNUOL69j2Ia6peDscN9nw/rTBKs9bsGiLk5ecKVJHxCoqjmcz7yCldTvbMWKh2/C21OF67hwGdgpCpyWmmNg96Frbx1QQf7vfIwwXf3p74X+oZBA6FazCdNzqr4dhuc0eGB62F6aNcFpSgV+ze5xsi22ME/DcFcPaj+cbD27J2pQ9VgurvMSNGN/u/CwlMKjPF3cqtkt1T+0Pz/WuuoxAWhcPqHnFmvRtERf2MAoTX/72u/hlthmOIB8LPKC6zsMTIKshbf+erX6ZhPIGkF5FigTAmNZBpf6929fU7ALCPMzUPJ61V2UHX5zV1TndrETskjSGLJ8d7UIvnS7z0L8m3fX1b847Lr1CKrQfyV4D4UlfgGB4KFI4GkH0eu6wwa7CTSIOXJAd4t4t6KfAsNS3yqbMZ0UZMWjOKT7g74Zf71YLObLNwu++rTGXr3xCZ6MYUNddsJuQoqCTGshvF0DwBUCRK+MYzcQdT3aWStCoTeUk8KKxeY+XPxu9m2xi5CFAtYSoA3SDIb2LpP7EB0X4l5yBZp8Nk6ixhdYsHkXIojr4Ns3pomnyHdpgu2jYSdscZaWRI3cb+aEo1iIW390JYqoEndgGhtcA+ldGLx5w3HOBC39PHNVSTeejwr4BkbPbkiRaJ/xWDOISEvs8FsIeCrT+ZIbV05F38YRSSngXBAKCLhAZDYIAAz/aNR0bip5jB7hSItHEJZdIypESmRkZ0TF/8QKHbd4TJMu3Nbmkl7M6MV8vrzDG3V6c0NDTFKIcNc0PMumjCLeXU+/YUnfhoSUBlGdBVGdB4HSwLe3kL4RbmJJArk9GZ240/wrNvyBd4dNapDI5wKCgG0BgZiu15rLzEcpHrfh4HCZZ3fbDFkjrx/lzfb6cctVNCkUkIZ20Bf4zxZTjh132JH585S8VvJGba/V1p8vjfxWVS+5bNEmzNg1rKD1W47NUUloR/8d2K8/98KqrzbBjYVg2wkPRNaH0Gjncwy0s5YsAn1r8xHGIaz7tU0V3Qlqa5Y4DZieOuXuGVBGyPDT35TmF6aAWcblGAyltm2aJa1uMQTRbUKPCMa+FG5LIcz8Gw+fvdsZ7uPrxsEKAx20MX3f22KUPuo2D0Ekic8uC6zVqEDP7JRdlNMIGBAl2x28A24oaV37uCSFowlbn9o02M9iawU+CJf260N7XuOuiBgLYlkpMBJYX8jUA4RkXGCjqQR8jpYOTk37TtyqbtwVnz7+9o1F4fb6SFaVfK4pPxXk8mqtr7prZpht3ZF0al/Arwa9GBur9Up7RF7qvwi7W4t6SmM+QAzzrJM807pHFaG8ce17WTRkv6kKbJY3VecDdRwZvFfLvq2PjZO+gswd6vgh9IakG/2nCMBo+++M17SVGDd6cHgx7vS4VAFhLGn3eY7+gLo/RlUPMpdia6JQqxtoI+wxwrev1BAn13K13ehrLDlJy2o7O28g0Q8Mm6rsnknr5C16e3/BGOsuRPN7c5cXRyzwafM9oKdvxVIS7LbpGgYjdRu80TW+Ub/QZv7KH/UCWq103lrvtumr0KdNRBtCMbzYsWf/IaPCXNDgZ4yw0I83/Ng17i0X1lUrZsWNBD5KjIPARjsHme34uxd93WS61tuma16l9nJYoSBxd1Syt5LjpkqPA3Uds4j4YzzFiEPrmLWGmiS7YbKlnQd/Z85qICI3uO8kf9CGxzfs5Tog/j27/IF2IYhB9yVXQQwa0x2pKJVcJ6N+CjjCvCGWOElax7JKLMWhSBEFyLNjGrYELJjm90ZYkU6sw5op3vh2H+XT6mn9arNmE7wZKKx2L5jz1xRuWhLwPbhe7FYQOoDQpw8eI9ZXhm4N59pSby0VSFLsZC1oXq4eSRyCsbIPhXElteYzoaO3M/swjC2QU6MnNmW7Gok58arfbLPezqYEZnLZ5MxpM0b8tE4WqxEDCk3b5gQR+uL+lf81Gqks9XVSpod1A8CrgUhf6sgauomsDKsyPJv98BDeWrsr9yN6dvtWWUc9knrLZvuqAPkyH965X4adx1kpYCNOipmPvtApMpJvGQpqjLpdLhbXFIBCFr/0qnKNKG7mmf5BoZd5IO03U0YBpc4Rmtp0KlBVNkyN3TQmK9TPHIJx9IEc5hAglDyEwRQPbM80uVLsRoSFSDJFavxYlUywCa6YUPr7DKyjzFvAsFIPqXoMPSJjfSak2zj3zuwVpbbb2kw9H9Zt/Hdvz3XpDv6gMaKPKu0QF/mN29AbiOMxcT9DAjcdaBoivMdKASlufJtewj0SP/MXhTtIcWsnKWhvhRUc7OBiUZs36XFbqcD5gC0NsxJb3Pqen32LpYdGQbx3kLWy5DawjDWFG6YDmrjWFxms5Aa0HGsZQLJu7tPOFa/MQg4WBGT2c1zGNxDwgyXxNrAzAV5+jesgC6Dd3RX9I1ysGmPniJIJ0Hfk+7gVTcWTpGcqeLgr/Ptsl7VjFFbzMEnlHvvN0rimVrg7/HZVtk0B+pXGTpzJ9EifKx2KCs41jttKxs/CYfYi42pqeYuf6V5w2EYSuKPIHzvd/sid2PCyiy74sOVm8tJr6mbIm75NmriYZosLseaduWqw486vXgwBoae3LhO3RJO3MDruABjTDSeXL5OmMh19wWlfy/T9EN09/KCkC+8ni7nMGKMVVmsg3cD6HbvMDALE7+yUyhwEC4ZKXJO/6VbYrtY+0c+sl+ip/Ll0fdd1bDppY+4GJPDxAIrPu7jW0Qp9xXI+JT1tzNWXqTv+RNa0Qe3cz50gfuG+NN6BnEXdbrlv/+P7f4uWr91pGO4xfaKmaV4fvLn2gj9f28zlRpMb3YJ3ATSKJ+JDX/t1iIluB1tK/2AP6pABTLvftZhm6fZYvoK4VOGSLY3dfrv8NL0WproTDahEDcbXmOfRia21YbTI4PqITjnw7XpcQtmMUrLTM6Rw/DMtHqiTdaA6ntPlxFcMctx4yW9v9NvNpZPqDp+VmcmrwRjZ0taXEHpFMnWEofERRvPGnxDpz4e8iYT9wvWP9RWcyvfNgTzZN8EQzKAmYQMew5joHhxSjZIwYc10e+HAm33mpfrLGmH3eUSGB+A3rb6QkTdEh0kRB36BqSO5vth2SszU3VoPwyxkNgm7KYiJc1e2B4JTPN8T9cX/w7qJX75Qgzh9/2U1jJ8ywp01/I5R/4dV97wTitj+Rmxdzym3vpc2XcK6KbjzCNPNuV3SItQTc5HKcKGlgxOxT9dbFk6Ya0uATM/wWdvun+2G5WiR5ZUpG5R5eUiHQ/hVrinKlaH5/zuC76o90fgRnyovAV9dpfSJZxhFSRFHkX8HchFCtJkkEUoITaxxZhOCzB4hK0hweyuStnq5YQWt5GWey+2F1AtK41Lv77mzGXscV5Dhwuv3EdTpVQW26J1b0jeGj5eZ/wDBKBODoBBb6P9WJfx2cWZPLi9dXPv2wt6Yu11cHLz55szO2PVukXthE/sLNgil0xhLXcPv1Dbdhvrjo87+o/SRO3fpAxJYgcGb0JoycOjjI8alIDMn+NDJXz4oE4kChBdGK8GsnoDCTgUblKEC/3+Wqa11WwE4Vfo8AkGjqQD2KPdkZy6CAs0TG3d9hbw5/q8AHWvfLsS3C3G7XGzGYnsKh6381MnrixqYXIPOEUtxZe11tw34FGi9C/mDBtf+ar9GMOYTNZqsezroN2iL0BD4+1Qthjw4LJvymH3zoZfilxeWpaTB8fdpKpvAlflqNbxdyMNobZ9b9X1qfX8bwYs+00xzNJyR3P2eljYuvfS9OWd7cmjiSUsOjdodOZr3X+vs7hpuCG/qurncJikuYqQ7dPik6WKv674ZMKZrrqFR/XD+rFDfh/dE1rKVJtVqPu2JmTg4KrlFgleJ6Pwp00QN3ErgR7O4L8/6r4I0YPN/BVHTC6AckReNIiAnitABRhF+XY2O8Or/AW5E1kW9SwAA"
  },
  "electric_full_song.py": {
    "sha256": "5629f87436e2168a80fe5e34327d41a12ebc3ffc1ac23d914c54ba21d2c55c25",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAji4UKZVzfQd+oZ9kn6XXWABgpTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ4yqCkaIqpAgKEYg4TaSosjgNIhmJ0zQOZmIp40zm3sHB50IWolxKIb+oosR5mUoSGJdXSalWsEQSCZksFKxxcn0xFA+qXKZVKVRZiEgFiySFaaE4fX/xXXFwPB69hjOFQeaJK5hRpFUeShGmSSm/lEOBJ1mlkYxhSBz/KJIUhsCe4bJK7nHzNBewXBku4cM7cBznQK2yNC9FkC+yIC+k+Q7TbGN+/w5rmN+roFya32lxMM/TlcigLVYzoZuvrSFFNcvyNJRFUbds6p94f16h3GQAJt18A3/H8jJYySILwvpEVaUi8/shWNftX1U2V7E8OPh48ulMTMXrV+Px+OD649nN2Sf4fDwQ8McpwqVcBf5a5oVKE2cijofCURH8cKTG8NEMMOxAM0EQe2D3opTZ0fr4DXZKZ8iLRXKtQokjwioKcEqWy1DplZ05kEL5+hW2q6Qo82olkxI7zAaFlFEBDbfjO73iAi4XJKH0C0AcrvxXb0zT5zKX1A7nwClvsDmd4xaRX6Z+mFXQ+imvpF4JaKPU57gogBZLoLU5YBsuXm5ioENDz3gYAfuWQS6CMExXWZAoPKknTkSUb4YiCcoqD2KRxVV4D8uUSF4ObWP9cZBiRRjLII83QEdArRVvG5RlEN4D0xRLRFSSllLksH+APIF0HwAnzWJkHmAWsQzyVZqosPB6NjkRBZGFWFWFAhjR8WPkGpUADyZpvtKNR/pOuVwAx8G68zSO0wekfuTDTAVJ+l3RswXvD/cGusTReMJokwQrPBKg/ung4CCSc+DKIkyBkPw8TUsX6WIoAP+AqVxG00sA0mDCuHCcc4XXFLNcRQvg06UM75G5DZMjSGZpei8WcToL4gL5EyipAiJdS4AmUEOS4FkMU8MKcBaHD49bA4Ujv9ExBl4uAeVr6Q6ofxXk94B2GOI6ICLmalGMAECyLIiaPeRrJMcyTeNiBLBa+XIdxCMQTV62IUINc5WV0CfLKvO1yMI+3gChARNU5KII0LfGPzmMzwEvcexSlxgBMa3ggKrwkVndAWAlpzbEnz4oL6rmDTSFKhBGAoHarE4r6nvXQwee/AIUHIFYzt0uJPS6uFTveenMgQJwfuQ7nuV5mrtz5+PV1SexCQADkcpUrJZiVt0HSYNJxuxEPOKKT86gCwJspjbAZAQ8XkrEB8BTnwzBAAQaAZ0CICIVlh5KxHu5KVxNW4QrEM7AmgPrzDiTIAHzXF5iaJYaAepINTh20w1guPhVJjIPUBMdfUCiKrRQ2/pzyBM9JE3XQTWyCjIfBZ5/ONKrD4adYSAxfa2NfB7jHzqDDqRVm2os0HhBxK0N/hhMuQyiDQDulm5sX72Z3L4GbGJIz1nLJErz0cnp2dENyvNj783Ig8b1aKaSUbYBZkwcizjbK6Ec2LsUEUOWqgQ4pasyRsS5XhHMZSmTIs0Le6M7Jo1lCuqE6ALkpIxcvi1csrlczRuxTFw9YSB+FscW020TsHMSgfCRM8B4FnSJ1hMXhRJI4f67i8tPwM3JAki7COJgCf8vKwNjNdmWlvaf7wXKCu93gIALNOKCyhuaSw1qBBI76FZQfHgXc3GgQkk8riUswcxnvLgoZTUBcQvACdv2ItaBXte50eKL54GAkA5umxYeSZ7pVDigmHl7xyaFA0tg6Mk1zvYDXH9pgT2TcbUSASAhw//99gGh6Yl/BhAnKEYKGEOytTYHwQoQs3QG++4FuROpWnkUo+MxMB2Ynb5R7v68imMfedZT2SaZDUUcxJUAvQyWpliQCCiU57RwwxfVKCDdvbSBD4abhwRxOwHkAScCll2Dh44GQblgd2s14gC98+rzHK07WwqTEYFmnZdmQOE4m3qBsPKZMyBLG6yFdGLLc5eavIUsaUEQa4Cfoahbk3AJ+hPQ22otglX2oCJQHoOB+AdQjmg4DsUr+K9XJfwGoDPoPfuSAYRBMaFlIVMwNu/ffRXXpx+Of9iW/c059H0H+vrA1mjogAz8Kn1wHxYuy0owPcAOlyVCDwypKdh6UYXCOk2mx/CxCFQyHXt/HYLG/OLXXeCnNBYHnDBWoSrRLVkcaVkM3gfb0AUcPU3AsphtwBoC7fqQour9EcTKIgg3IgbTBxwP2H4DpAokUNbWBhnqQASIZMK3sddz+fdK5ZIaaeQaAUZmB12juQTfoH36mtfcINm4YEODxYyWsLsGzQcrDlAOIh+2u1w4Heg2tLQH9RB0ToBRwdxVQAlrNjLWqCX4SG3BTr4QnE/8JMZmhTF8mJOJn6atk+I3gNoeitfB5uMtqWATDfiAgs/EG/5sI/anqb3HT4hM0ji0NKr28fD4TtNW4015IC/cWxvyrjOfrzK5QN5zjtBrBC8V5fLRGv8v8Sj0yY3KYRZlwhv02wDOEfkrNA4PTkuXuqVGIDYGc7Q0nHUK8k5OH/HwT86uRYNcL4GMx9NDPBR5TEfhhD2qcOUXxz/EUo9lzhjcDVmNTdHlYagAkVZg+0+NVNEjQfDg+oa8glnh6pFHNfBJg+qBezA4d97KMEXvvqaFlSqA2sLlhMgfdX4kHk3vExA6UMgj7/fUFrPcqCUBmK9o2dX8D9o5q3r53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnRa7PcctQZdfGpVcY8ByDdhEsw7XCJxdx29GtG6wT+fWUmP7ig096e7jMYqFjuR4Vki0kE3CCz3Wh2UaowCJkT7yBjYmRgUWBIj5qNAwSnO1UEkQ+8ugQDeJycIwOoMxmafQgyTiYSChcC3JQuzlMyFuixgYNZMsYzpyBZz8Bx98lFzJnT6FM0c/vZyai9LEFGWHo51RSyYBq6PuYk7XrL4OFNg5Mbq9hDEXr3Krl3Xubp165bsB8DmRj23MtdFWL0d4axb/CfEHLa5NbDWhNeN+Rqx64zd7EXtDRtkWtkQ6ZwPhR0IyGmyEYzgiaGfYCQ4LQhxPslJJBS7AC3igOdr3Ag9mabO9R+QwIqxGkRqJdAd2KJ6rkW7ySyjzrDRURvThre4jhZYaOqcFYWrIkU4/vZ+eB2BY83AMD6HhztPQJjRk6oHRxwZGmK5WeAmwLf9ExbatxO72gXFSH+N7OEePtrtrnZUHkfYzyg/sMzrIN2k5gpPeo6Pt9Zbb+o986Ub98RI20QMJHZKKQznEo7Upc/ic5qsVH5EqBt9KJM4domcvaYL+5b1Z3Xa0MFLrS86hqXCLVms61UdhF9LwGpiivPf3Zu9jsA8sAwHcMwrqTzsTD814ms5o4bCGDkI1BM3tDTn3eDSMGsulMUww7Ay09sDhD3o4nlJGdiAMvz3wIIEDg1Xhuq/Ij2GHBiPFl1eXZ0hniAoM8eZATQijRtE9LFHozOI0vBeTaX0iD2MQmqKIgCdmNzIvuCcPHlya2YqyaVjunOD8+9gBwLoG5ofiH/V5EOYKKYJR8aij9r+1gvYa0/D12Ej7SRtvGLpEXod2+vtphyk6k0FJgXjQNU6UPiSthhAILNJf/fMfghzPSmOcE2SRI/Ku7ItQFkYlYKyhFOXY8uhebka0vFjJMoiCMvCcuyemm9p8cxvishc0cVu7bbB7qjYxIzPtsT8XshOU9c2bQxQE/BPMMVhU/zS0hvhsdtQDW1YIE/ZT39Jwl2aqnrQPDIMnze8ajEQ2tQlUr9QyivCcyIlbG2n+3MK1vuXW+A6bYmAhlj6GIWAMM6FDgsNn8odW+uzZwQK/LSVNNsrfQ+kyiXzNS80+uqHvLjZjNPwkI3uPxjw2d0eXDjpAu+0hU5WgUaAJzaCk5f2YRu3/yC8yBNvG5zAEBYwaJ+iQ8iog3cgextyrSaZNOZXWRD0+SkzHosECg3SIRmdVwUCB73jDqVWhVeoRiK0AMyqULRudX79+NdJBEszt5CqyEi18Ckw5cFaRUxTYZOxCCiFyW+1K1adl3wNc+ToBGEEHHB9tTJ6EUSLXzhZSZKr53qca/4VgZ+9Haprc044Nx5tpFWmH+Pb5lewQVBksL4NVv3+pw5dg/69VniZ0CuNmbnfRPHvotG+UfTIrWeQ67879d59/8U9PTt+RcoPvfz371Hx/uH7/y8np384u3+qv06vL84tf3158dCxLPS08vdktLo0xTWtzbjtoD/SqDIPxrj7A1fn5+4vLs6lzDPt8+nhyeXN+9fHD2ccbu0drthYM3T3xazxygKmVVtDVA+zKvET9bgdW++ZrXWpy7GkeLm1jkBo8TONgSLt2GJ6La59+fnsiqqRxMDBtRzFujTlSc6fXn0cfrm/EPIjjWRDeO62z6NyIB/ZaFAPFYkwXJIv+tEfiXLhaYbfBPjiDufpAexWYwz7ndPs7Xsa1lvROQolw0V3WFVH2+JRDX4Fc81mkIiX7TTLHLWQ8BykU5AsQs4eH9w/4azuXxe3MwboygLiXiwMmveC0a020KNE+TyEQ1E47EoEn8SKskgAqZQzqIoPWsCzHYzu316Ya4U7YG71Vn0boUY9+OzmbNkUK87gqlpY/0YliF1VGGdVnwdUFVBvYAB5/HUifbqFBy9DSoS2z3/b1dmMUoNHGfjsCoskIRhmCwvm/8G+LgkyNj+5xuwt0CcfH8IHvv4hAavCZSTvB1Do6G/SUKgbU85ePoGsNpxAim/77T4F/uOO2rhBBice0Z7VtzdJY6Rxs/y26Z8aN6EuXLWzhAgZ0cEAj/yJOQZCgM6DtBBfcoHWqwDEz9Urf2VVSbHxTkdQR20ec0PC0wkHJ4dnJF9h3KxdjND4ZzS0PzzSyUbDQifS0HddomrvDdLCkLzwClofMlVaF+pxZHCSu2VIrBdsGGmL5UHe60VCYU/g9nRkrEbMKht382hyYmvIlZDDi6amhBy6B2uH2AJm2CpB03KddwzTFLBWsMnUKEkF+RhZb13a0oAjAsy6ijcimRRstQGwAo06NmGs063S/ZtWkN9XAbIQXW5j1bXcDRHABDcjBRM0BivWGewpstJjT0D7oAFFDDyho2gBjWFPgtCGBPqTWq3Whj/VjyNVVDjKXgif51HKkuEfjo8w3jbjQ5Gc4jvm8iwYMw2Ql1kTKM/qJCETtjOrNWmuLMMHrKKti6szBjgBOGvKM6dx5JL1AXygs0fzy/SdwH6mpVVmD+pTNQpRIsXX2byQs44ugneFTpGsrIRumFbIcZWCP7TzMizIr/x+JuzoJF4AYXU3RK2SHdPpIh38adrJz+2OU1rp/OCOn5t1cHJhDdIh9fswVU2cEzi/VXSDn5Wupq2eDsKyVEK/OKzoGfSAH7oOF1N7oH0YRuhhtpJOz4dbloX6M5YQJhTqGTdWoD+dcK/ngrbLXiELXWakvVH9hRuJ3a5Clm/8PCKNm8r2p3RZmYzWDg8UAU1ppxq3H//TqvrvmbpyDlAK4omzmX1QtU7ht/01XYWjI1kUqoJKKKi69rypzEII1bNFf3gJ4Gw8taA/23Nbeby5lhLpge0cbUb24s5BH8QRdfuz9m8rOsT6oBhXf1/lCkTqOvGJqrx5/ce2/PTt/D4z3lstbwOxVa9m22+ZUEBqDHFuz83uo4YxmJ4IS7GY9keOszf5m2rD+ZTiljaCJHZ5xbW1dF4gaFTRqxXcsU2e0JV9bt7BHFgyHvkXG47F/Qjj3sUJ6PNJkYYZqpTXSf+sAoCkpIoVSY7YtBhzH+ZuUmQhEsQKNITDUi7qXPFkGhwjmmEy4uvowwqVg/aYWfxUUVD2PpTFhUIFmryND3SzADhprFZShP09WYOE+R0049k8hIhM96cN9t2yyRcJWteJOUmNSp9EaGxqrUkf0uuUNh7sqHIbG0tfWsSbOnkUwVRNkLgitGZbXTXQ5sFX5C3zdN3Hwx6Jfp1cfPlx8uhnuLa/4hvAXayYyWn2tAqwp4IksFjJq7VDH47AKPEJL0KYqHYi3kMUaFsnoMi3PQVdGpppF53tx2hzbwcri6cbMYquXVrKqLY2J23owwVxoVW6biVatN+b/rGIDexCmmXwMSuhiPI6w7jMTWg9yTPAkUvM5lrrzaxIwGAo6JDhwJsvRKCKtYyMOJv7ZhfGTbsG5CdaZbQfbOLKvRxXnJs/aejikHxHoNz8c1k7zzbcl5LXQoyQbmeSYzGE5D3uZVzCU4bHcN9YFGEKe0AshLwKpij9cxtuW+nWMY+Zj+lrhPM1Qt+SDYZqt41FtZVme9bC2d+UAFb6BKhp3Vye6sGwW74Xvp7A8UiVVWlFwrHlHRRm7SGZgRCMrN2+okBrM8ynx28kZVWuUiuKhoD7IHfHEuyCPHgAF4KCURPzm0QpsUJD1Em7wHRZookRicp3r0kpJiPacPRmWHvXMqOzx5KycWH+dWSuoMRSdak27nhV36LpwGKBIaCKZBdN2vudbb2BxjlYFk54Y59y5fn9yOaEHb49m49vv2rmy7+6eCoNTCk//ev1Z1E+qno988rEOujtfXZ6J65Obm4m9c7PnxHs9f9LVysNnDqcj6EyhzXO9msZ2nNGqlEHrfLtcXtdfk0r0Sdj4vqUZyaE44pwbWfxHOMtpcgs8AHDWsv8HTXSQSIXCYyz4ZD73yR3DWGdDN7Uia6pXQA9OdyZ7tt2KhvxuHT6wCX9Q/LJnd1P2pbXnVnjNUkRuX9huy5YdtJSUTaTWSIr+a0nK0X9jAkUOpvnw2UYz/NYB7V1gjRqMPH5x0XlKIgt9eipBt0L7TEgqWachxx6bc8JOGFTc2vt2fGe9JaI3Qv1xzC273LEwgwY6RVGNTVI/RYLfQ0qOtkCmZ5i8KaljMvJ3pE9wBKx020nB3+GImrW6nc8B1E6K6APxzuzgoM2g5oohqR0LC6Acyqb6C9BH5PDiEnAKWZZUDXJ36/Ag1G99vTzVirZLg2JUyM0TTtaqVnhePx7FRPyw79Xorjej/eUrdlFBDctOvQFuWYL/Q4F81JuxXKTtp61B3HqCurUNGNTgZ/n8RlN9NVuSNYKrg6RD98pqAhcLGLr0G0hwT//6ZjTeoehZBV+9VMFCmq4nmyDx1QEji0jwXm6I+ujBAJmL0DLUn2AwGkR5Ch/TuYNnX4ycsG5vvcag/YCH80Ibrj12K9tZI1Pt1GIiJh8tc+gtMZ0ZX1XhozGrd4ZFAH6hvmqmOu4OQBjlIJjTFfE2jNKFewSp5y73OakJ14q34kIj2lnURG8pL/bed0rifi+/XwaThYdqEH8QtugHPQvk2XRJGgY0ewsCF8EHnwY8lA+AW+NbMNCgmHvENKXFmugRt1ZrSXndw0K+NUzz0ZZE675B6ZtEvMZzWulQx9SX0426KyEph1qQ7l0h7LkI2zx4EQIjwxA2Gbz8TVSjmrCchnXSSOceRDvjYADbLk5vC/VuFdfd1uudbnlYHaeyqtIGezXFy5ds1St3Fm1Vk909C7ALUjL4Nm3BT0rQu7O1uZVDHNgqoklP1I96a+2CTioaeiiwm+P3RVKbc7dq0+52bLV/sZ7eVnx212bNY7bnbogx2K3NOjH256/U78I0NtqwFlXTbYWNKTyiYl9T8VT/vSvabOlgnauzrZytXmdn2DqTwT2/uQZeRNsOjKzZBhzF1oI7R71kYZ1m2b9ue9DuZYkMmRGmW7y0TT9g8CMmd01ox/Z37cn0htWP1vtImFnmIN1N25uhIS+mjenzJPOt3msnEXWw26t8YRa1n3b/FxlUHeDeZe93HSBbYuuprRhW2z3jT/LOLAVv5vWq8W+FNEY7dgX+LalCQK4BjLWxIGPFX8RHIAHFz1zqzGLNmyOW/GzRSDPZ+4NZ6BddRz96h7OYiDyVtqA/ZspcvJN8Qbb2NfW4keQIJMbNfT9KQ/DvrZn47yj4gZ7iNv49Usj0mnIKsGUAjD7dFSPQ/+JEcXt8t3fl+t+ZqNfeO5xjCtY5TFjU8vl37USFy3oymTPNLcZ7Z1rPzezJ+I80TZ2rlSo7b+1MlGwVfFGraoVPZYwFu2MHHaqiWhpCCijkFPO4cKn9M+soTM9EfcSaCG4+X6NHeNMqhdHr0l+4ssm5om+DdYm8gV2T0Sq5pjFIHnbihBoBVd1/hkO2fAKzQ1E/H+MXVPpEkq0dQyLYb5DtdBUz3KNOGdUnGtrLN4cyuSPubMcsubGpXjfZJGrWH4NORM+5IYk6qTny1ljG6Pv+h+C6BOw2J0CWVVgHyDKWnAefSyN99KWJkw/+Bzrm61BTTAAA"
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
